# Swiss Legal Retrieval — Phase 1: BM25 Baseline

**Zero external dependencies** — uses only pandas + numpy (pre-installed on Kaggle).  
Works fully offline. ✅

**Metric:** Macro F1 (citation-level, averaged across queries)

In [ ]:
# No pip install needed — only standard Kaggle packages
import pandas as pd
import numpy as np
import re
import os
import math
from collections import defaultdict

print('Libraries loaded ✅')

## ⚙️ Config

In [ ]:
DATA_DIR    = '/kaggle/input/llm-agentic-legal-information-retrieval'
OUTPUT_PATH = '/kaggle/working/submission.csv'

TOP_K      = 5      # citations to return per query
USE_COURT  = False  # skip the 2.4GB court file for this baseline

## 📂 Load Data

In [ ]:
print('Loading test queries...')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'  → {len(test_df)} test queries')

print('Loading laws corpus...')
laws_df = pd.read_csv(f'{DATA_DIR}/laws_de.csv')
print(f'  → {len(laws_df):,} law snippets')

if USE_COURT:
    print('Loading court considerations (2.4GB)...')
    court_df = pd.read_csv(f'{DATA_DIR}/court_considerations.csv')
    corpus_df = pd.concat([laws_df, court_df], ignore_index=True)
else:
    corpus_df = laws_df

print(f'Total corpus: {len(corpus_df):,} documents')
corpus_df.head(3)

## 🔤 Tokenizer

In [ ]:
def tokenize(text: str) -> list:
    """
    Lowercase + regex tokenizer.
    Keeps German umlauts and dots/slashes for citation IDs like 'BGE 139 I 2 E. 3.1'
    """
    if not isinstance(text, str):
        return []
    return re.findall(r'[a-z0-9äöüß./]+', text.lower())

# Quick sanity check
print(tokenize('What are the inheritance rights under Art. 462 ZGB?'))

## 🏗️ BM25 Index (pure NumPy — no external packages)

BM25 formula:  
`score(q, d) = Σ IDF(t) * (tf(t,d) * (k1+1)) / (tf(t,d) + k1*(1 - b + b*|d|/avgdl))`

- **IDF**: how rare is the term across all documents (rare = more informative)
- **TF**: how often does the term appear in this document
- **k1, b**: tuning parameters (standard values: k1=1.5, b=0.75)

In [ ]:
class BM25:
    def __init__(self, corpus_tokens: list, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.N  = len(corpus_tokens)  # total number of documents

        # Document lengths and average length
        self.doc_lens = np.array([len(doc) for doc in corpus_tokens], dtype=np.float32)
        self.avgdl    = self.doc_lens.mean()

        # Build inverted index: term → list of (doc_id, term_freq)
        # Also track document frequency (df) = how many docs contain this term
        self.inv_index = defaultdict(list)   # term → [(doc_id, tf), ...]
        self.df        = defaultdict(int)    # term → doc count

        for doc_id, tokens in enumerate(corpus_tokens):
            term_counts = defaultdict(int)
            for t in tokens:
                term_counts[t] += 1
            for term, tf in term_counts.items():
                self.inv_index[term].append((doc_id, tf))
                self.df[term] += 1

    def idf(self, term: str) -> float:
        """Inverse document frequency — rare terms score higher."""
        df = self.df.get(term, 0)
        if df == 0:
            return 0.0
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def get_scores(self, query_tokens: list) -> np.ndarray:
        """Return BM25 score for every document in the corpus."""
        scores = np.zeros(self.N, dtype=np.float32)
        for term in set(query_tokens):  # deduplicate query terms
            if term not in self.inv_index:
                continue
            idf_val = self.idf(term)
            for doc_id, tf in self.inv_index[term]:
                dl   = self.doc_lens[doc_id]
                denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf_val * (tf * (self.k1 + 1)) / denom
        return scores

print('BM25 class defined ✅')

In [ ]:
# Build the index
# Combine citation string + text so citation keywords also help matching
print('Tokenizing corpus...')
corpus_texts   = corpus_df['citation'].fillna('') + ' ' + corpus_df['text'].fillna('')
corpus_tokens  = [tokenize(doc) for doc in corpus_texts]

print('Building BM25 index...')
bm25 = BM25(corpus_tokens)
print(f'Index ready! {bm25.N:,} docs, avg length {bm25.avgdl:.1f} tokens ✅')

## 🔍 Retrieve Citations

In [ ]:
print(f'Retrieving top-{TOP_K} citations per query...')
predictions = []

for _, row in test_df.iterrows():
    query_id     = row['query_id']
    query_tokens = tokenize(row['query'])

    if not query_tokens:
        predictions.append({'query_id': query_id, 'predicted_citations': ''})
        continue

    scores    = bm25.get_scores(query_tokens)
    top_k_idx = np.argsort(scores)[::-1][:TOP_K]
    top_k_idx = [i for i in top_k_idx if scores[i] > 0]  # drop zero-score docs

    top_citations = corpus_df.iloc[top_k_idx]['citation'].tolist()
    predictions.append({
        'query_id': query_id,
        'predicted_citations': ';'.join(top_citations)
    })

print(f'Done — {len(predictions)} predictions')

## 💾 Write Submission

In [ ]:
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH} ✅')
submission_df.head(10)

## 📊 Local Validation on val.csv

In [ ]:
val_path = f'{DATA_DIR}/val.csv'
if os.path.exists(val_path):
    val_df = pd.read_csv(val_path)

    f1_scores = []
    for _, row in val_df.iterrows():
        tokens  = tokenize(row['query'])
        scores  = bm25.get_scores(tokens) if tokens else np.zeros(bm25.N)
        top_idx = np.argsort(scores)[::-1][:TOP_K]
        top_idx = [i for i in top_idx if scores[i] > 0]
        pred    = set(corpus_df.iloc[top_idx]['citation'].tolist())
        gold    = set(str(row['gold_citations']).split(';'))

        if not gold and not pred:
            f1_scores.append(1.0)
        elif not gold or not pred:
            f1_scores.append(0.0)
        else:
            tp = len(gold & pred)
            p  = tp / len(pred)
            r  = tp / len(gold)
            f1_scores.append(2*p*r / (p+r) if (p+r) > 0 else 0.0)

    print(f'Val Macro F1 : {np.mean(f1_scores):.4f}')
    print(f'Per-query F1 : {[round(s,3) for s in f1_scores]}')